### 05 - Preparation des donnees de classification
#### HumanForYou - Attrition ML

**Objectif** : preparer des donnees **train/test** pour la classification supervisee, en reprenant la logique du workshop (split + encodage + normalisation), mais avec les donnees du projet.

- **Entree**: `data/processed/kmeans_clusters.csv`
- **Sorties**: `data/processed/attrition_train_prepared.csv`, `data/processed/attrition_test_prepared.csv`

---

#### Classification binaire : prediction de l'Attrition

**Nature du probleme** :
- **Variable cible** : `Attrition` (0 = reste, 1 = quitte) — variable **binaire categorielle**
- **Type de tache** : **classification supervisee** (pas une regression)
- **Precision terminologique** : les notebooks 05-07 portent le prefixe "Regression" par heritage du workshop, mais la tache reelle est bien une **classification binaire**

**Pourquoi ce n'est pas une regression lineaire** :
- Une regression lineaire predit une **valeur continue** (ex: salaire, prix). Ici, la cible est binaire.
- La **Regression Logistique** (utilisee dans les notebooks suivants) est un modele de **classification** malgre son nom. Elle estime la probabilite P(Attrition=1) via une fonction sigmoide.
- Les metriques utilisees sont des metriques de classification : **Accuracy, Precision, Recall, F1-score, AUC-ROC** (et non R² ou MSE).

**Strategie de preparation** :
1. Split stratifie 70/30 (conserver la proportion d'attrition dans train et test)
2. Encodage one-hot des variables categorielles
3. Standardisation `StandardScaler` (fit sur train, transform sur test — pas de data leakage)

#### 1. Imports

Cette section charge les bibliotheques necessaires pour:
- lire les donnees,
- faire le split train/test,
- normaliser les features numeriques apres encodage.


In [1]:
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


#### 2. Chargement des donnees projet

On recharge la sortie du notebook 04 (`kmeans_clusters.csv`) qui contient deja:
- les traitements metier des notebooks precedents,
- les features badgeuse,
- le cluster KMeans,
- la cible `Attrition`.


In [2]:
DATA_PATH = os.path.join('..', 'data', 'processed', 'kmeans_clusters.csv')
assert os.path.exists(DATA_PATH), f'Fichier introuvable: {DATA_PATH}'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Colonnes: {len(df.columns)}')
print('Distribution Attrition:')
print(df['Attrition'].value_counts(dropna=False).sort_index())
df.head()


Shape: (4410, 28)
Colonnes: 28
Distribution Attrition:
Attrition
0    3699
1     711
Name: count, dtype: int64


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeID,Gender,JobLevel,...,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating,avg_work_hours,cluster
0,51,0,Travel_Rarely,Sales,6,2,Life Sciences,1,Female,1,...,1,0,0,3.0,4.0,2.0,3,3,7.373651,2
1,31,1,Travel_Frequently,Research & Development,10,1,Life Sciences,2,Female,1,...,5,1,4,3.0,2.0,4.0,2,4,7.718969,0
2,32,0,Travel_Frequently,Research & Development,17,4,Other,3,Male,4,...,5,0,3,2.0,2.0,1.0,3,3,7.013240,2
3,38,0,Non-Travel,Research & Development,2,5,Life Sciences,4,Male,3,...,8,7,5,4.0,4.0,3.0,2,3,7.193678,1
4,32,0,Travel_Rarely,Research & Development,10,1,Medical,5,Male,1,...,6,0,4,4.0,1.0,3.0,3,3,8.006175,2


#### 3. Verifications de continuite

On verifie la coherence minimale avant de preparer les donnees de modelisation.


In [3]:
required_cols = ['Attrition', 'avg_work_hours', 'cluster']
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f'Colonnes attendues manquantes: {missing}'

assert df.isna().sum().sum() == 0, (
    'Des NaN subsistent alors que 02/03/04 ont deja nettoye les donnees.'
)
assert set(df['Attrition'].unique()).issubset({0, 1}), (
    'Attrition doit etre binaire 0/1.'
)

print('Validation des pre-traitements precedents: OK')


Validation des pre-traitements precedents: OK


#### 4. Split + encodage + normalisation (logique workshop)

Etapes:
1. separation `X` / `y`,
2. retrait de l'identifiant technique (`EmployeeID`) si present,
3. split stratifie (`test_size=0.30`),
4. encodage one-hot,
5. normalisation `StandardScaler` sur train puis application sur test.


In [4]:
TARGET = 'Attrition'

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).copy()

if 'EmployeeID' in X.columns:
    X = X.drop(columns=['EmployeeID'])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

# Encodage categoriel (compatible object/string/category)
X_train_enc = pd.get_dummies(X_train_raw, drop_first=False, dtype=float)
X_test_enc = pd.get_dummies(X_test_raw, drop_first=False, dtype=float)

# Alignement strict des colonnes train/test
X_train_enc, X_test_enc = X_train_enc.align(
    X_test_enc,
    join='outer',
    axis=1,
    fill_value=0.0,
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_enc),
    columns=X_train_enc.columns,
    index=X_train_enc.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_enc),
    columns=X_test_enc.columns,
    index=X_test_enc.index,
)

train_prepared_df = X_train_scaled.copy()
train_prepared_df[TARGET] = y_train.loc[X_train_scaled.index].to_numpy()

test_prepared_df = X_test_scaled.copy()
test_prepared_df[TARGET] = y_test.loc[X_test_scaled.index].to_numpy()

print(f'X_train_scaled: {X_train_scaled.shape}')
print(f'X_test_scaled : {X_test_scaled.shape}')
print('Distribution train Attrition:')
print(train_prepared_df[TARGET].value_counts(normalize=True).sort_index())
print('Distribution test Attrition:')
print(test_prepared_df[TARGET].value_counts(normalize=True).sort_index())


X_train_scaled: (3087, 46)
X_test_scaled : (1323, 46)
Distribution train Attrition:
Attrition
0    0.838678
1    0.161322
Name: proportion, dtype: float64
Distribution test Attrition:
Attrition
0    0.839002
1    0.160998
Name: proportion, dtype: float64


#### 5. Export des jeux prepares

Les fichiers exportes sont ensuite utilises tels quels dans les notebooks 06 et 07.


In [5]:
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_output_path = os.path.join(PROCESSED_DIR, 'attrition_train_prepared.csv')
test_output_path = os.path.join(PROCESSED_DIR, 'attrition_test_prepared.csv')
feature_cols_path = os.path.join(PROCESSED_DIR, 'attrition_feature_columns.csv')
scaler_stats_path = os.path.join(PROCESSED_DIR, 'attrition_scaler_stats.csv')

train_prepared_df.to_csv(train_output_path, index=False)
test_prepared_df.to_csv(test_output_path, index=False)

pd.DataFrame({'feature': X_train_scaled.columns}).to_csv(feature_cols_path, index=False)
pd.DataFrame({
    'feature': X_train_scaled.columns,
    'mean_train': scaler.mean_,
    'scale_train': scaler.scale_,
}).to_csv(scaler_stats_path, index=False)

print(f'Saved: {train_output_path}')
print(f'Saved: {test_output_path}')
print(f'Saved: {feature_cols_path}')
print(f'Saved: {scaler_stats_path}')


Saved: ..\data\processed\attrition_train_prepared.csv


Saved: ..\data\processed\attrition_test_prepared.csv
Saved: ..\data\processed\attrition_feature_columns.csv
Saved: ..\data\processed\attrition_scaler_stats.csv


In [6]:
assert train_prepared_df.isna().sum().sum() == 0, 'NaN dans train_prepared_df'
assert test_prepared_df.isna().sum().sum() == 0, 'NaN dans test_prepared_df'

assert 'Attrition' in train_prepared_df.columns and 'Attrition' in test_prepared_df.columns
assert 'cluster' in train_prepared_df.columns, (
    'La feature cluster issue du notebook 04 doit etre preservee.'
)

print('Validation OK pour 05_Regression_Preparation.ipynb')


Validation OK pour 05_Regression_Preparation.ipynb


---

#### Mini-conclusion — Notebook 05

**Ce qui a ete fait** :
- Chargement du dataset enrichi (notebook 04) avec features RH, badgeuse et cluster
- Split stratifie 70/30 preservant la proportion d'attrition (~16 %)
- Encodage one-hot des variables categorielles (alignment strict train/test)
- Standardisation `StandardScaler` fittee sur train uniquement (pas de data leakage)
- Export des fichiers `attrition_train_prepared.csv` et `attrition_test_prepared.csv`

**Points de vigilance** :
- Le scaler est fitte **uniquement** sur le train pour eviter toute fuite d'information du test
- La feature `cluster` est preservee comme variable predictive supplementaire
- L'`EmployeeID` est retire avant modelisation (aucun pouvoir predictif)

**Prochaine etape** : Notebook 06 — Entrainement et evaluation des modeles de classification